# magics

> The `%%prompt` cell magic: solveit-style prompt cells for plain Jupyter. Running a prompt cell sends everything *above* it (using `yasi.dialog.extract_dialog`) plus the prompt itself, and inserts the tagged response as a new markdown cell below.

The magic is registered automatically when you create a `JupyterChat`. Cells below the prompt cell are not sent, so you can run prompts in the middle of a notebook.

In [ ]:
#| default_exp magics

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from IPython.core.magic import Magics, magics_class, cell_magic

from yasi.dialog import extract_dialog, notebook_upto_prompt

@magics_class
class YasiMagics(Magics):
    """Registers the %%prompt cell magic, bound to a `JupyterChat` instance."""
    def __init__(self, shell, jc):
        super().__init__(shell)
        self.jc = jc

    @cell_magic
    def prompt(self, line, cell):
        """Send the dialog up to (and including) this prompt cell, insert the response below"""
        jc = self.jc
        notebook = jc.get_current_nb()
        notebook, found = notebook_upto_prompt(notebook, cell)
        messages, warnings = extract_dialog(notebook,
                                            tag_system=jc.tag_system,
                                            tag_user=jc.tag_user,
                                            tag_assistant=jc.tag_assistant,
                                            all_cells=jc.all_cells,
                                            max_output_len=jc.max_output_len,
                                            hide_tag=jc.hide_tag)
        for warning in warnings:
            jc.create_new_markdown_cell(warning)
        if not found:
            # unsaved or edited prompt cell: still send the prompt itself
            messages.append({"role": "user", "content": cell.strip()})
        response_text = jc.chat_client.chat(messages,
                                            model=jc.model,
                                            max_tokens=jc.max_tokens,
                                            temperature=jc.temperature,
                                            presence_penalty=jc.presence_penalty)
        jc.latest_response = jc.chat_client.latest_response
        jc.create_new_markdown_cell(f"{jc.tag_assistant}\n\n{response_text.strip()}")

def load_prompt_magic(jc, # A JupyterChat instance the magic sends dialogs with
                      shell=None # IPython shell, defaults to the running one
                     ):
    """Registers the %%prompt magic in the running IPython shell"""
    if shell is None:
        from IPython import get_ipython
        shell = get_ipython()
    if shell is not None:
        shell.register_magics(YasiMagics(shell, jc))

The magic only needs a duck-typed `JupyterChat`, which keeps it testable without a JupyterLab frontend:

In [ ]:
from types import SimpleNamespace

class FakeChat:
    """Minimal stand-in for JupyterChat"""
    tag_system, tag_user, tag_assistant = '#| chat_system', '#| chat_user', '#| chat_assistant'
    all_cells, max_output_len, hide_tag = True, 2000, 'yasi-hide'
    model, max_tokens, temperature, presence_penalty = 'test-model', None, None, None
    latest_response = None
    def __init__(self, notebook):
        self.notebook = notebook
        self.created_cells = []
        self.chat_client = SimpleNamespace(latest_response='raw-response',
                                           chat=self.fake_chat)
    def fake_chat(self, messages, **kwargs):
        self.sent_messages = messages
        return 'A fine answer.'
    def get_current_nb(self): return self.notebook
    def create_new_markdown_cell(self, content): self.created_cells.append(content)

notebook = {'cells': [
    {'cell_type': 'markdown', 'source': ['# Notes']},
    {'cell_type': 'code', 'source': ['x = 42'], 'outputs': []},
    {'cell_type': 'code', 'source': ['%%prompt\n', 'Why 42?'], 'outputs': []},
    {'cell_type': 'markdown', 'source': ['below the prompt, must not be sent']},
]}

fake = FakeChat(notebook)
magic = YasiMagics(shell=None, jc=fake)
magic.prompt('', 'Why 42?')

assert [m['content'] for m in fake.sent_messages] == ['# Notes', '```python\nx = 42\n```', 'Why 42?']
assert fake.created_cells == ['#| chat_assistant\n\nA fine answer.']
assert fake.latest_response == 'raw-response'

# unsaved prompt cell: prompt still gets appended
fake2 = FakeChat(notebook)
YasiMagics(shell=None, jc=fake2).prompt('', 'A brand new question')
assert fake2.sent_messages[-1] == {'role': 'user', 'content': 'A brand new question'}
print('magic tests passed')

Usage in JupyterLab (the magic is auto-registered by `JupyterChat`):

```python
jc = JupyterChat(openai_base_url="https://openrouter.ai/api/v1")
```

then in any code cell:

```
%%prompt
What is money? Explain it like my mate from Oz.
```

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()